In [1]:

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the dataset
df7 = pd.read_csv("Detailed_Polling_Data.csv")
df7.columns = df7.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# Wipe out pre-existing calculated column copies to avoid value re-assignment errors
cols_to_clear = ['Margin_Percentage', 'Winner_Votes', 'Runner_Up_Votes', 'Margin_Of_Victory', 'Winner_Party', 'Cluster_ID']
df7 = df7.drop(columns=[c for c in cols_to_clear if c in df7.columns], errors='ignore')

# 2. Party Array Definitions matching your exact current parameters
core_parties = [
    'Dravida Munnetra Kazhagam',
    'All India Anna Dravida Munnetra Kazhagam', 
    'Bahujan Samaj Party', 
    'Naam Tamilar Katchi', 
    'Tamilaga Vettri Kazhagam',
    'All India Puratchi Thalaivar Makkal Munnettra Kazhagam', 
    'Tamizhaga Vaazhvurimai Katchi'
]
df7[core_parties] = df7[core_parties].fillna(0)

station_col = 'Polling Station No.'
building_col = 'Location and Name of the Building in which Polling station is located'
area_col = 'Polling Area'

# Isolate the 2 explicit independent fields present in this specific dataset
independent_candidate_cols = ['Independent', 'Independent.1']
existing_ind_cols = [c for c in independent_candidate_cols if c in df7.columns]
df7[existing_ind_cols] = df7[existing_ind_cols].fillna(0)
df7['Total_Independent_Votes'] = df7[existing_ind_cols].sum(axis=1)

# 3. Calculate true total votes for normalization
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['Total_Independent_Votes'] + df7['NOTA'].fillna(0)
df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()

# 4. Explicit Abbreviation Mapping to keep feature streams completely unique
party_abbreviations = {
    'Dravida Munnetra Kazhagam': 'DMK',
    'All India Anna Dravida Munnetra Kazhagam': 'AIADMK',
    'Bahujan Samaj Party': 'BSP',
    'Naam Tamilar Katchi': 'NTK',
    'Tamilaga Vettri Kazhagam': 'TVK',
    'All India Puratchi Thalaivar Makkal Munnettra Kazhagam': 'AIPTMMK',
    'Tamizhaga Vaazhvurimai Katchi': 'TAVAK'
}

share_cols = []
for party in core_parties:
    party_label = party_abbreviations[party]
    col_name = f'{party_label}_share_pct'
    
    if col_name in df7.columns:
        df7 = df7.drop(columns=[col_name])
        
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

df7['independent_share_pct'] = (df7['Total_Independent_Votes'] / df7['Total_Calculated_Votes']) * 100

# Metric calculations
df7['Winner_Votes'] = df7[core_parties].max(axis=1)
sorted_votes = np.sort(df7[core_parties].values, axis=1)
df7['Runner_Up_Votes'] = sorted_votes[:, -2]
df7['Margin_Of_Victory'] = df7['Winner_Votes'] - df7['Runner_Up_Votes']
df7['Winner_Party'] = df7[core_parties].idxmax(axis=1)
df7['Margin_Percentage'] = (df7['Margin_Of_Victory'] / df7['Total_Calculated_Votes']) * 100

feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']
X = df7[feature_cols].copy().fillna(0)

# 5. Extract and Scale features for the ML model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Profile Breakdowns
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
print(df7.groupby('Cluster_ID')[feature_cols].mean().round(2))
print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
# Note: Rename your cluster strings in this dictionary based on your terminal outputs later
cluster_names = {0: "Cluster_0_Target", 1: "Cluster_1_Target", 2: "Cluster_2_Target", 3: "Cluster_3_Target"}

for cluster_num in range(optimal_k):
    target_cols = [station_col, building_col, area_col, 'Winner_Party', 'Margin_Percentage']
    valid_target_cols = [c for c in target_cols if c in df7.columns]
    
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][valid_target_cols]
    filename = f"Dataset_7_Cluster_{cluster_num}_{cluster_names[cluster_num]}.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated cleanly for all 4 clusters.")

/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (



--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            DMK_share_pct  AIADMK_share_pct  BSP_share_pct  NTK_share_pct  \
Cluster_ID                                                                  
0                   26.34             32.89           0.18           3.27   
1                   34.39             34.35           0.48           2.06   
2                   33.26             29.47           0.11           2.52   
3                   24.71             47.27           0.15           2.02   

            TVK_share_pct  AIPTMMK_share_pct  TAVAK_share_pct  \
Cluster_ID                                                      
0                   36.01               0.28             0.22   
1                   27.89               0.10             0.07   
2                   33.94               0.06             0.07   
3                   25.04               0.11             0.08   

            independent_share_pct  Margin_Percentage  
Cluster_ID                    